In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 1 &mdash; Introduction</h1></center>

Every geophysical result you have seen (a seismic image, a gravity map, an earthquake location) began as a recorded signal that was
windowed, filtered, resampled, deconvolved and transformed before anyone interpreted it. This course is about those steps: what each one
does to a signal, why it works, and what it can break. The two examples below come from my own research, a teleseismic image of the
subducting Juan de Fuca plate and a microseismic monitoring survey, and in both the final result depended on choices made during processing.
The module ends with a map of the course and a short guide to how these notes are organized.

# Outline
- [1.1 Why Digital Signal Processing?](#1.1-Why-Digital-Signal-Processing?)
- [1.2 Example: Imaging Cascadia with Teleseismic Data](#1.2-Example:-Imaging-Cascadia-with-Teleseismic-Data)
- [1.3 Example: Microseismic Velocity Inversion](#1.3-Example:-Microseismic-Velocity-Inversion)
- [1.4 Course Outline](#1.4-Course-Outline)
- [1.5 How to Use These Notes](#1.5-How-to-Use-These-Notes)
- [Microquestions](#Microquestions)

<a name="1.1"></a>
## 1.1 Why Digital Signal Processing?

In my experience, (geophysical) digital signal processing (or "DigSig") represents one of the most important sequences of steps that we geophysicists use when taking geophysical data recorded in the field and generating our desired geophysical analysis outputs (i.e., models, images) that are used in publications, reports and theses. 

However, there are many challenges that early-career geophysicists face when starting out on this path.  In particular, with geophysical data (like in other disciplines) there are many many many different **processing strategies** and **algorithms** to choose from, the **order of processing workflow** must be developed, and this usually involves selecting values for scores of different **parameters**.  Is there a right or wrong way to process the data? How does one establish a judicious, well thought out, and computationally efficient workflow? In many ways it might seem to someone coming in (and indeed the same is felt by many other geoscientists that use our end products!) that Figure 1.1(a) must be true.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
fig, (axa, axb) = plt.subplots(2, 1, figsize=(12, 5.4), gridspec_kw=dict(height_ratios=[1, 1.15]))

def box(ax, x, y, w, h, text, fc, ec, tc='k', fs=11):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle='round,pad=0.02,rounding_size=0.08',
                                fc=fc, ec=ec, lw=1.8))
    ax.text(x, y, text, ha='center', va='center', fontsize=fs, color=tc)

def arrow(ax, x0, x1, y):
    ax.annotate('', xy=(x1, y), xytext=(x0, y), arrowprops=dict(arrowstyle='-|>', lw=1.8, color='k', mutation_scale=16))

# (a) the "magic black box"
for ax in (axa, axb):
    ax.set_xlim(0, 12); ax.axis('off')
axa.set_ylim(0, 2)
box(axa, 1.6, 1.0, 2.6, 1.3, 'Field\ngeophysical\ndata', 'white', C['time_domain'])
box(axa, 6.0, 1.0, 3.6, 1.3, 'Magic black box\nof geophysical\ndata processing', '#222222', '#222222', tc='white', fs=12)
box(axa, 10.4, 1.0, 2.6, 1.3, 'Top-notch\ngeophysical\nresult', 'white', C['freq_domain'])
arrow(axa, 2.95, 4.15, 1.0); arrow(axa, 7.85, 9.05, 1.0)
axa.set_title('(a) What processing can look like from the outside', fontsize=13)

# (b) the box opened: the CASC93 processing chain (caption of Figure 1.3)
axb.set_ylim(0, 2.3)
steps = ['Automated\nwindowing', 'Band-pass\nfilter', 'Polarity\ncorrection', 'Source\ndeconvolution',
         'Regularize\nto a grid', 'Migration\n(imaging)']
xs = np.linspace(1.0, 11.0, len(steps))
for k, (x, s) in enumerate(zip(xs, steps)):
    box(axb, x, 1.0, 1.55, 1.15, s, C['passband'], C['time_domain'], fs=10.5)
    if k:
        arrow(axb, xs[k-1] + 0.80, x - 0.80, 1.0)
axb.set_title('(b) The box opened: each step is an operation on a signal that we can analyze', fontsize=13)
plt.tight_layout(); plt.show()

<b>Figure 1.1.</b> (a) What digital signal processing might seem to some at this point. Just say NO! to the Black Box. (b) The same box opened, for the teleseismic example of Section 1.2: the processing chain described in the caption of Figure 1.3, from windowing to migration. Every box is a signal-processing operation of the kind studied in this course. <i>(Python-generated.)</i>

One of the major goals of this class is to strip away some of the **magic black box** nature of geophysical data processing, and give you a better conceptual idea of how to analyze your geophysical data. Ideally, this will help you to design better data processing workflows that allow us to achieve our geophysical data processing goals (e.g., noise removal, signal enhancement, interpretable geophysical result).

In the sections below I present two examples that (hopefully!) show the value of applying  digital signal processing to geophysical field data.

<a name="1.2"></a>
## 1.2 Example: Imaging Cascadia with Teleseismic Data

The goal of the CASC93 experiment was to obtain recordings of distant (so-called teleseismic) earthquakes that would provide us with some idea about the lithospheric structure in a seismically quiescent zone located in west-central Oregon.  The acquisition of this data set on a linear array of 40 three-component seismometers created a fantastic opportunity to apply the concepts of 2D migration to study the top 120 km of lithosphere!

<div style="display:flex; justify-content:center; align-items:flex-end;"><div style="text-align:center; margin:0 10px;"><b>(a)</b> CASC93 experiment location<br><img src="Fig/EQMIG1.png" width="450"></div><div style="text-align:center; margin:0 10px;"><b>(b)</b> Earthquake locations<br><img src="Fig/EQMIG1B.png" width="400"></div></div>

<b>Figure 1.2.</b> (a) Location of the CASC93 seismic experiment involving 40 three-component broadband seismic stations deployed over 300 km in Oregon. (b) Locations of the large earthquakes (i.e., > 5.5 Mb) used for imaging that originated $30^\circ <\Delta<90^\circ$ from the array. <i>(Source: Rondenay, Bostock and Shragge (2001), <i>Geophysical Journal International</i>.)</i>

<center><img src="Fig/EQMIG2.png" width="400"></center>

<b>Figure 1.3.</b> Example of a processed earthquake arrival recorded on a multicomponent seismic array. Recorded earthquake signals were subject to much digital signal processing: automated windowing (based on short-time versus long-time signal magnitude estimates), bandpass filtering (within the source signal band), polarity filtering (to account for directivity of source), estimation and deconvolution of source signatures (to estimate the impulse response), and regularization into a regular grid for plotting (for ease of applying migration operators). <i>(Source: Rondenay, Bostock and Shragge (2001), <i>Geophysical Journal International</i>.)</i>

<center><img src="Fig/EQMIG4.png" width="500"></center>

<b>Figure 1.4.</b> Stack of all 40 processed earthquakes after undergoing the imaging procedure. The interpretation of the structure shown in this figure is the Juan de Fuca Plate subducting beneath the North American continental plate. Note that this final step involved applying full-wavefield migration, which is also a digital filter. <i>(Source: Rondenay, Bostock and Shragge (2001), <i>Geophysical Journal International</i>.)</i>

Figures 1.2&ndash;1.4 are extracted from <i>Rondenay, S., M.G. Bostock and J. Shragge, 2001, Multiparameter two-dimensional inversion of scattered teleseismic body waves 3. Application to the Cascadia 1993 data set: Geophysical Journal International, 106, 12, 30795-30807.</i>

<div class="alert alert-info">
<b>Synthesis &mdash; geophysical data as the running example.</b> The image in Figure 1.4 is a 2D signal in exactly the sense of the
gravity and magnetic anomaly maps you worked with in GPGN 328 (POTE1): a measured quantity sampled over position. In GPGN 328 the physics told
you what produced the map; in this course the question is what the recording and processing did to it. The same question is asked of every
data example in these notes.
</div>

<a name="1.3"></a>
## 1.3 Example: Microseismic Velocity Inversion

You have no doubt heard about the link between subsurface fluid injection activities (i.e., fracing) and induced (micro)seismicity into prolific shale formations of very low permeability.  In most cases, the magnitudes of induced microseismic events are quite low (< 1.0 Mw), and it is helpful for the operators to know where and when these events occur because they provide important details as to how the stimulation increases fault connectivity and overall permeability.  

There are now a number of specialized microseismic operators that deploy tens-to-hundreds of multi-component seismic stations (see Figure 1.5) that are sufficiently sensitive to record **very weak microseismic events** (e.g., $-1.25 < M_w < 0.25$) occurring at 1.0-2.0 km depth.

<center><img src="Fig/MICRO00.jpg" width="500"></center>

<b>Figure 1.5.</b> Example field deployment of 192 3C sensors over the Marcellus Shale formation in Ohio. The overall area of investigation is 6$\times$6 km$^2$, while the white box shows the 1.5$\times$1.25 km$^2$ area of subsurface stimulated at approximately 2.0 km depth. <i>(Source: Witten and Shragge (2017), <i>Geophysics</i>.)</i>

<div style="display:flex; justify-content:center; align-items:flex-end;"><div style="text-align:center; margin:0 10px;"><b>(a)</b> Two windowed raw microseismic events<br><img src="Fig/MICRO1.png" width="500"></div><div style="text-align:center; margin:0 10px;"><b>(b)</b> Events processed into P- and S-waves through DSP<br><img src="Fig/MICRO2.png" width="350"></div></div>

<b>Figure 1.6.</b> (a) Example of a weak (upper panels) and a very weak (lower panels) microseismic event, where the three subpanels represent the vertical, easting and northing components. (b) The same two events after significant digital signal processing: data windowing (again based on long-term versus short-term signal averages), wave-mode separation (to isolate P- and S-wave contributions), band-pass filtering (to within expected range), envelope estimation (to handle S-wave polarities), and second pass of bandpass filtering (to return signals to zero mean). <i>(Source: Witten and Shragge (2017), <i>Geophysics</i>.)</i>

<div class="alert alert-info">
<b>Note:</b> The vertical, easting and northing traces in Figure 1.6 are the three components of a ground-motion vector, so
"wave-mode separation" and "polarity filtering" start with the MATH 213 operations of projecting a vector onto new axes. Microquestion 1.3 works
one rotation by hand.
</div>

Usually, these data sets are acquired in frontier exploration areas that have no 3D (or even 2D) seismic data, which are essential for generating information on the local 3D velocity structure.  (In fact, commonly the only velocity constraint in the area is a single borehole located a few kilometers away that might only be logged in the interval of interest!)  Thus, the question is: **how can you locate source of a very noisy microseismic event if you do not know the 3D velocity structure and you do not know at what time and where it occurred?**  

When at the University of Western Australia (UWA), my PhD student Ben Witten and I developed a method to take the processed microseismic signals and go through a process of **adjoint-state inversion** to jointly estimate the 3D velocity structure and optimize microseismic event locations.  Figure 1.7 shows the results before (left two panels) and after (right two panels) applying the velocity inversion method.  The upper and lower panels show the P- and S-wave velocity structure, respectively.  Overall, through the inversion process there is a significant improvement in the consistency of event locations - and potentially imaging of a stimulated fault.

<div style="display:flex; justify-content:center; align-items:flex-end;"><div style="text-align:center; margin:0 10px;"><b>(a)</b> Before inversion<br><img src="Fig/MICRO3.png" width="400"></div><div style="text-align:center; margin:0 10px;"><b>(b)</b> After inversion<br><img src="Fig/MICRO4.png" width="400"></div></div>

<b>Figure 1.7.</b> The initial (a) P-wave and (b) S-wave velocity structure used to generate initial location estimates (white stars). The (c) P-wave and (d) S-wave velocity structure used to generate the final location estimates. Much more consistent locations are now observed, and (almost) all events fall within the expected interval. The quality of the final result is heavily dependent on the digital signal processing choices made during the preprocessing stages. <i>(Source: Witten and Shragge (2017), <i>Geophysics</i>.)</i>

Note: Figures 1.5&ndash;1.7 are extracted from <i>Witten, B. and J. Shragge, 2017, Microseismic Image-domain Velocity Inversion: Marcellus Shale Case Study: Geophysics, 82, no. 6, KS99-KS112.</i>

<a name="1.4"></a>
## 1.4 Course Outline

This introductory course on geophysical digital signal processing covers a lot of fundamental mathematical and numerical topics that you will need in your career as a geophysicist or more broadly when playing a quantitative role in the physical sciences.

Table 1.1 lists the modules; each is described below.

In [ ]:
import pandas as pd
modules = [('00', 'Syllabus'), ('01', 'Introduction'), ('02', 'Signals, Systems and Processing'), ('03', 'Complex Numbers'), ('04', 'Fourier Series'), ('05', '1D Fourier Transforms'), ('06', '2D Fourier Transforms'), ('07', 'LTI Systems, Convolution and Correlation'), ('08', 'Signals as Vectors, Systems as Matrices: the SVD'), ('09', 'Discrete-Time Signals'), ('10', 'Digital Sampling and Reconstruction'), ('11', 'Discrete Fourier Transform'), ('12', 'Windows and Spectrograms'), ('13', 'Wavelet Transforms'), ('14', 'Z-Transform'), ('15', 'Practical Filtering')]
df = pd.DataFrame(modules, columns=['Module', 'Title'])
display(df.style.hide(axis='index').set_properties(**{'text-align': 'left'})
        .set_table_styles([{'selector': '', 'props': [('width', '70%')]},
                           {'selector': 'th', 'props': [('text-align', 'left')]}]))

<b>Table 1.1.</b> The modules of GPGN 404. Each module is one notebook, <code>NN_Title.ipynb</code>. <i>(Python-generated.)</i>

<b>Module 01 &ndash; Introduction.</b> The purpose of this module is to emphasize how fundamental DSP has been to your instructor in his career, and to highlight the key topics that will be investigated in this course.

<b>Module 02 &ndash; Signals, Systems and Processing.</b> This short module introduces the vocabulary used throughout the course: signals; continuous, discrete and digital signals; systems; and processing.

<b>Module 03 &ndash; Complex Numbers.</b> This module provides a refresher on the manipulation of complex numbers and functions, which are fundamental building blocks for the material developed later on in the course.

<b>Module 04 &ndash; Fourier Series.</b> This module starts to build up the Fourier machinery by examining continuous, periodic time series. We calculate Fourier spectra that are a very helpful tool for examining signals in a different light.

<b>Module 05 &ndash; 1D Fourier Transforms.</b> In this module we examine both the analytical and numerical properties of the 1D Fourier transform of continuous (non-periodic) time series. We also look at the Fourier transforms of a number of important analytic signals, as well as those of instrument responses, earthquake recordings and seismic data.

<b>Module 06 &ndash; 2D Fourier Transforms.</b> This module extends the 1D Fourier analysis presented in Module 5 to 2D images. Concepts of 2D spatial filtering are introduced. We investigate the wavenumber structure of 2D magnetic and gravity data sets.

<b>Module 07 &ndash; LTI Systems, Convolution and Correlation.</b> This module introduces the concepts of linear time-invariant (LTI) systems, including the operations of convolution, correlation, autocorrelation and deconvolution.

<b>Module 08 &ndash; Signals as Vectors, Systems as Matrices: the SVD.</b> This module writes a sampled signal as a vector and a linear system as a matrix. The singular value decomposition (SVD) then shows what a system amplifies, what it loses, and why deconvolution can be unstable. Applications include the pseudo-inverse, low-rank image compression, and SVD filtering of seismic data. No linear algebra beyond matrix–vector products is assumed.

<b>Module 09 &ndash; Discrete-Time Signals.</b> This module introduces a number of important concepts regarding discrete time series including: continuous vs discrete vs digital signals; deterministic vs stochastic signals; and characteristics of LTI systems.

<b>Module 10 &ndash; Digital Sampling and Reconstruction.</b> This module explores the key consequences of performing digital sampling (i.e., aliasing), discusses strategies to avoid aliasing, and how to reconstruct analog signals from properly sampled time series.

<b>Module 11 &ndash; Discrete Fourier Transform.</b> This module looks at how we can take 1D/2D Fourier transforms that we defined in a continuous fashion in the modules above and looks at how they can be posed in discrete systems, including the frequency axis of the DFT and spectral leakage.

<b>Module 12 &ndash; Windows and Spectrograms.</b> This module looks at how we can go beyond Fourier transforms that span the entire time series to those that provide more information on the "local" frequency information.

<b>Module 13 &ndash; Wavelet Transforms.</b> This module extends the time–frequency analysis of Module 12 with wavelets: the continuous wavelet transform, with scalograms of a chirp and of the 2011 Tohoku earthquake record, and the discrete Haar wavelet transform as a pair of filters. Applications are denoising and the compression of 1D signals and 2D images.

<b>Module 14 &ndash; Z-Transform.</b> This module extends the discrete-time Fourier transform from the unit circle to the whole complex plane, giving the algebra used to design and analyze digital filters.

<b>Module 15 &ndash; Practical Filtering.</b> We finish the course by looking at how Z-transforms can be used to set up FIR and IIR filters that can be used to do a whole bunch of useful things such as lowpass, highpass, bandpass and bandreject filtering of 1D signals.

<a name="1.5"></a>
## 1.5 How to Use These Notes

Each module is a Jupyter notebook that you should read and run before class. A few conventions
are used throughout:

* <b>Colored boxes carry meaning.</b> Blue boxes are notes, synthesis boxes that connect a result to earlier modules or prerequisite courses,
  and proofs. Yellow boxes flag common misconceptions and hold the microquestions. Green boxes (<i>Distractions</i>) are optional enrichment.
* <b>Microquestions.</b> Each module ends with three microquestions. The worked answers are hidden in collapsible <i>Show answer</i> blocks:
  try the question first.
* <b>Code.</b> The Python code is hidden by default so that the text and figures read cleanly. Use the <i>Show code</i> button at the top of
  each notebook to see it, and rerun cells to experiment with the parameters. Animations play once their cell has been run.
* <b>Software.</b> The notebooks run in the <code>dsp2026</code> Python environment; the README in the course repository explains how to install
  it and select it as the notebook kernel.
* <b>Generative AI.</b> Generative AI tools are allowed with disclosure on labs and the project, but not on quizzes; see the Syllabus (Module 00).

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 1.1 &mdash; Triggering on an arrival (STA/LTA)</b>

The automated windowing in Figure 1.3 and Figure 1.6 compares a short-term average (STA) of signal power with a long-term average (LTA). A trace contains noise of power 1 (RMS amplitude 1, arbitrary units). An arrival adds power 16 (RMS amplitude 4) for 0.5 s. The STA window is $T_s = 0.5$ s long and the LTA window is $T_l = 10$ s long; both end at the same sample.

<b>(a)</b> When the STA window lies entirely inside the arrival, compute STA, LTA and the ratio STA/LTA.

<b>(b)</b> Write STA/LTA for an arrival of added power $S$ as a function of $S$, $T_s$ and $T_l$, using the average value of a function over an interval (MATH 213). What does the ratio tend to as $T_l\to\infty$, and why is a long LTA window useful?

<b>(c)</b> A noise burst of RMS amplitude 2 lasting 0.5 s also occurs. With a trigger threshold of 3, is it detected? What does this say about what STA/LTA actually measures, and why are the very weak microseismic events of Figure 1.6 hard to pick?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 1.1</b></summary>

<b>(a)</b> STA $= 1 + 16 = 17$. LTA $= [(10 - 0.5)(1) + (0.5)(17)]/10 = (9.5 + 8.5)/10 = 1.8$. STA/LTA $= 17/1.8 = 9.44$.

<b>(b)</b> STA/LTA $= (1 + S)/(1 + S\,T_s/T_l)$. As $T_l \to \infty$ the ratio $\to 1 + S = 17$: the LTA becomes a pure noise estimate, so the ratio measures the arrival against the background rather than against itself.

<b>(c)</b> $S = 4$: ratio $= 5/(1 + 4 \times 0.05) = 5/1.2 = 4.17 > 3$, so the burst triggers. STA/LTA detects a jump in power, not a seismic arrival. A weak event with RMS amplitude 1.5 ($S = 2.25$) gives only 2.92, below threshold: such events need further processing before they can be located.
</details>

<div class="alert alert-warning">
<b>Microquestion 1.2 &mdash; Wavelength sets the detail</b>

The CASC93 images use teleseismic P waves with dominant frequency 1 Hz in rock of P-wave speed 7 km/s; the image spans 120 km depth. The microseismic survey records P waves of about 50 Hz in rock of P-wave speed 3.5 km/s, at 2 km depth. Take a quarter wavelength as the thinnest layer an image can resolve.

<b>(a)</b> Compute the wavelength and $\lambda/4$ for each experiment ($v = f\lambda$, PHGN 200).

<b>(b)</b> How many wavelengths fit between the surface and the target in each case? Why is the CASC93 slab in Figure 1.4 drawn as broad bands while the microseismic events in Figure 1.7 are located to tens of meters?

<b>(c)</b> A student proposes band-pass filtering the CASC93 data to 3.5 Hz to get $\lambda/4 = 0.5$ km resolution. What is wrong with this plan? (Hint: the caption of Figure 1.3 says the band-pass was applied "within the source signal band".)
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 1.2</b></summary>

<b>(a)</b> CASC93: $\lambda = 7/1 = 7$ km, $\lambda/4 = 1.75$ km. Microseismic: $\lambda = 3.5/50 = 0.07$ km $= 70$ m, $\lambda/4 = 17.5$ m.

<b>(b)</b> $120/7 \approx 17$ wavelengths vs $2/0.07 \approx 29$. The target depths in wavelengths are similar, but the absolute wavelength is 100 times larger for CASC93, so its image features are kilometers wide while microseismic locations are constrained to tens of meters.

<b>(c)</b> A filter can only reweight frequencies that are present in the recording. Teleseismic P energy above a few hertz has been attenuated over thousands of kilometers of travel, so a 3.5 Hz band-pass would mostly pass noise. Processing cannot create signal outside the source band.
</details>

<div class="alert alert-warning">
<b>Microquestion 1.3 &mdash; Rotating three-component data</b>

A P wave arrives from back-azimuth $\phi = 30^\circ$ (measured clockwise from north) and moves the ground with unit amplitude along that direction, so a three-component station records north $N = \cos\phi$ and east $E = \sin\phi$. Radial and transverse components are $R = N\cos\phi + E\sin\phi$ and $T = -N\sin\phi + E\cos\phi$.

<b>(a)</b> Compute $N$, $E$, $R$ and $T$.

<b>(b)</b> Repeat the rotation with an assumed back-azimuth of $40^\circ$ (a $10^\circ$ error). What fraction of the energy $N^2 + E^2$ leaks onto $T$? Show that $R^2 + T^2 = N^2 + E^2$ for any rotation angle (MATH 213: rotation preserves vector length).

<b>(c)</b> A student says the rotation "removes noise" because $T \approx 0$ after rotating. Critique this, using (b).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 1.3</b></summary>

<b>(a)</b> $N = \cos 30^\circ = 0.866$, $E = \sin 30^\circ = 0.500$; $R = 0.866^2 + 0.500^2 = 1.000$, $T = -0.866 \times 0.5 + 0.5 \times 0.866 = 0.000$.

<b>(b)</b> With $40^\circ$: $R = \cos 10^\circ = 0.985$, $T = -\sin 10^\circ = -0.174$; $T^2 = 0.030$, i.e. 3.0% of the energy. $R^2 + T^2 = (N\cos\alpha + E\sin\alpha)^2 + (-N\sin\alpha + E\cos\alpha)^2 = N^2(\cos^2\alpha + \sin^2\alpha) + E^2(\sin^2\alpha + \cos^2\alpha) = N^2 + E^2$, since the cross terms cancel.

<b>(c)</b> Rotation only redistributes energy between components; it removes nothing. $T \approx 0$ for a P wave from the correct back-azimuth because the motion is radial. Noise and S-wave energy are rotated too, and a wrong back-azimuth puts signal on $T$. This is why the polarity and wave-mode steps in Figure 1.3 and Figure 1.6 need the source direction.
</details>